# 03 - Duplicate Detection: FDA Medical Device Master Data
## Project: Healthcare Device Master Data Governance
### Objective
Identify and link duplicate records across two MDM dimensions:
1. **Applicant Name Deduplication** — detect same company under different name variations using fuzzy matching
2. **Device Name Deduplication** — identify same device submitted multiple times

This notebook simulates real enterprise MDM deduplication workflows used in 
tools like Informatica MDM, Reltio, and Oracle Customer Data Management.

In [2]:
pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Import libraries
import pandas as pd
import numpy as np
from rapidfuzz import fuzz, process
import re

# Load dataset
df = pd.read_csv('/Users/harshithasunkara/Documents/Projects/Healthcare MDM/data/raw/pmn96cur.txt',
                 sep='|',
                 encoding='latin-1',
                 low_memory=False)

print(f"Dataset loaded: {df.shape[0]:,} records")

Dataset loaded: 99,216 records


In [4]:
 # ================================
# STEP 1: PREPARE APPLICANT NAMES
# ================================
print("=== PREPARING APPLICANT NAMES FOR MATCHING ===")

def normalize_name(name):
    if pd.isna(name):
        return ''
    name = str(name).lower().strip()
    # Remove common legal suffixes
    suffixes = [', inc.', ', inc', ' inc.', ' inc', ', llc', ' llc',
                ', ltd', ' ltd', ', corp.', ', corp', ' corp.', ' corp',
                ', co.', ' co.', ', gmbh', ' gmbh', ', lp', ' lp',
                ', l.p.', ' l.p.']
    for suffix in suffixes:
        name = name.replace(suffix, '')
    # Remove special characters and extra spaces
    name = re.sub(r'[^\w\s]', ' ', name)
    name = re.sub(r'\s+', ' ', name).strip()
    return name

# Apply normalization
df['APPLICANT_NORMALIZED'] = df['APPLICANT'].apply(normalize_name)

print(f"Original unique applicants: {df['APPLICANT'].nunique():,}")
print(f"Normalized unique applicants: {df['APPLICANT_NORMALIZED'].nunique():,}")
print(f"Reduction from normalization: {df['APPLICANT'].nunique() - df['APPLICANT_NORMALIZED'].nunique():,}")

# Show examples
print(f"\nNormalization examples:")
examples = df[['APPLICANT', 'APPLICANT_NORMALIZED']].drop_duplicates().head(10)
print(examples.to_string())

=== PREPARING APPLICANT NAMES FOR MATCHING ===
Original unique applicants: 23,272
Normalized unique applicants: 21,996
Reduction from normalization: 1,276

Normalization examples:
                         APPLICANT      APPLICANT_NORMALIZED
0                   Ohmeda Medical            ohmeda medical
1   Boston Scientific Scimed, Inc.  boston scientific scimed
2                   Urosurge, Inc.                  urosurge
3            Usa Instruments, Inc.           usa instruments
4                          Tornier                   tornier
5            Maet Industries, Inc.           maet industries
6                       Daig Corp.                      daig
7        Diagnostic Products Corp.       diagnostic products
8                 Urometrics, Inc.                urometrics
11         Inova Diagnostics, Inc.         inova diagnostics


In [6]:
# See the full impact of normalization across ALL companies
print("=== NORMALIZATION IMPACT EXAMPLES ===")

# Find records where normalization actually changed something
changed = df[df['APPLICANT'] != df['APPLICANT_NORMALIZED']][['APPLICANT', 'APPLICANT_NORMALIZED']].drop_duplicates()
print(f"Total names that changed after normalization: {len(changed):,}")
print(f"\nSample of changes:")
print(changed.head(30).to_string())

=== NORMALIZATION IMPACT EXAMPLES ===
Total names that changed after normalization: 23,272

Sample of changes:
                                             APPLICANT                               APPLICANT_NORMALIZED
0                                       Ohmeda Medical                                     ohmeda medical
1                       Boston Scientific Scimed, Inc.                           boston scientific scimed
2                                       Urosurge, Inc.                                           urosurge
3                                Usa Instruments, Inc.                                    usa instruments
4                                              Tornier                                            tornier
5                                Maet Industries, Inc.                                    maet industries
6                                           Daig Corp.                                               daig
7                            Diagnostic P

In [7]:
def normalize_name(name):
    if pd.isna(name):
        return ''
    name = str(name).lower().strip()
    # Remove legal suffixes - whole words only
    suffixes = [r',?\s+inc\.?$', r',?\s+llc\.?$', r',?\s+ltd\.?$',
                r',?\s+corp\.?$', r',?\s+corporation$', r',?\s+incorporated$',
                r',?\s+co\.?$', r',?\s+gmbh$', r',?\s+lp$', r',?\s+l\.p\.$']
    for suffix in suffixes:
        name = re.sub(suffix, '', name)
    # Remove special characters and extra spaces
    name = re.sub(r'[^\w\s]', ' ', name)
    name = re.sub(r'\s+', ' ', name).strip()
    return name

# Reapply
df['APPLICANT_NORMALIZED'] = df['APPLICANT'].apply(normalize_name)

print(f"Original unique applicants: {df['APPLICANT'].nunique():,}")
print(f"Normalized unique applicants: {df['APPLICANT_NORMALIZED'].nunique():,}")
print(f"Reduction: {df['APPLICANT'].nunique() - df['APPLICANT_NORMALIZED'].nunique():,}")

# Verify the fix
test_names = ['Biosite Incorporated', 'Prime Pacific Health Innovations Corporation', 
              'Boston Scientific Scimed, Inc.', 'Urosurge, Inc.']
for name in test_names:
    print(f"{name} → {normalize_name(name)}")

Original unique applicants: 23,272
Normalized unique applicants: 21,769
Reduction: 1,503
Biosite Incorporated → biosite
Prime Pacific Health Innovations Corporation → prime pacific health innovations
Boston Scientific Scimed, Inc. → boston scientific scimed
Urosurge, Inc. → urosurge


In [8]:
# ================================
# STEP 2: FUZZY MATCHING
# ================================
print("=== FUZZY MATCHING FOR REMAINING DUPLICATES ===")

# Get unique normalized names
unique_names = df['APPLICANT_NORMALIZED'].dropna().unique().tolist()
unique_names = [n for n in unique_names if len(n) > 3]  # skip very short names
print(f"Unique names to match: {len(unique_names):,}")

# Find matches above 90% similarity threshold
matches = []
for name in unique_names[:500]:  # test on first 500 first
    results = process.extract(name, unique_names, 
                             scorer=fuzz.token_sort_ratio,
                             limit=5)
    for match, score, _ in results:
        if score >= 90 and name != match:
            matches.append({
                'NAME_1': name,
                'NAME_2': match,
                'SIMILARITY_SCORE': score
            })

matches_df = pd.DataFrame(matches).drop_duplicates()
print(f"Potential duplicate pairs found: {len(matches_df):,}")
print(f"\nSample matches:")
print(matches_df.head(20).to_string())

=== FUZZY MATCHING FOR REMAINING DUPLICATES ===
Unique names to match: 21,602
Potential duplicate pairs found: 117

Sample matches:
                        NAME_1                         NAME_2  SIMILARITY_SCORE
0              usa instruments                 sa instruments         96.551724
1            inova diagnostics             genova diagnostics         91.428571
2            life technologies           lifeloc technologies         91.891892
3            life technologies               led technologies         90.909091
4        acist medical systems            cas medical systems         90.000000
5        acist medical systems            is2 medical systems         90.000000
6            nobel biocare uas              nobel biocare usa         94.117647
7         medist international         meditech international         90.476190
8         medist international           medien international         90.000000
9                apple medical                applied medical       

In [9]:
# ================================
# STEP 3: CLASSIFY MATCHES
# ================================
print("=== CLASSIFYING DUPLICATE CANDIDATES ===")

# High confidence - 95%+ similarity
high_confidence = matches_df[matches_df['SIMILARITY_SCORE'] >= 95]

# Medium confidence - 90-94% similarity, needs human review
medium_confidence = matches_df[
    (matches_df['SIMILARITY_SCORE'] >= 90) & 
    (matches_df['SIMILARITY_SCORE'] < 95)
]

print(f"High confidence duplicates (95%+): {len(high_confidence):,}")
print(f"Medium confidence - needs review (90-94%): {len(medium_confidence):,}")

print(f"\n=== HIGH CONFIDENCE DUPLICATES ===")
print(high_confidence.to_string())

print(f"\n=== MEDIUM CONFIDENCE - NEEDS STEWARD REVIEW ===")
print(medium_confidence.to_string())

=== CLASSIFYING DUPLICATE CANDIDATES ===
High confidence duplicates (95%+): 26
Medium confidence - needs review (90-94%): 91

=== HIGH CONFIDENCE DUPLICATES ===
                                          NAME_1                                    NAME_2  SIMILARITY_SCORE
0                                usa instruments                            sa instruments         96.551724
19                          biocell laboratories                      biocoll laboratories         95.000000
23                   medtronic sofamor danek usa               medtronic sofamor danck usa         96.296296
26                                    rs medical                               srs medical         95.238095
27                                    rs medical                               qrs medical         95.238095
32                            alcon laboratories                         acon laboratories         97.142857
35                            depuy orthopaedics                         dep

In [10]:
# ================================
# STEP 4: SAVE DUPLICATE REPORT
# ================================

# Add confidence label
matches_df['CONFIDENCE'] = matches_df['SIMILARITY_SCORE'].apply(
    lambda x: 'HIGH' if x >= 95 else 'MEDIUM'
)

# Save to reports folder
matches_df.to_csv(
    '/Users/harshithasunkara/Documents/Projects/Healthcare MDM/reports/duplicate_candidates.csv',
    index=False
)

print(f"=== DUPLICATE DETECTION SUMMARY ===")
print(f"Total candidate pairs found: {len(matches_df):,}")
print(f"High confidence (auto-merge candidates): {len(matches_df[matches_df['CONFIDENCE']=='HIGH']):,}")
print(f"Medium confidence (steward review needed): {len(matches_df[matches_df['CONFIDENCE']=='MEDIUM']):,}")
print(f"\nReport saved to reports/duplicate_candidates.csv")

=== DUPLICATE DETECTION SUMMARY ===
Total candidate pairs found: 117
High confidence (auto-merge candidates): 26
Medium confidence (steward review needed): 91

Report saved to reports/duplicate_candidates.csv
